In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC
from sklearn.svm import SVC

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Read files

In [2]:
# trainingset

x1_train = pd.read_csv("/Users/kellyt/Documents/ml/HW2/train1_icu_data.csv")
y1_train = pd.read_csv("/Users/kellyt/Documents/ml/HW2/train1_icu_label.csv")

In [3]:
# testset

x1_test = pd.read_csv("/Users/kellyt/Documents/ml/HW2/test1_icu_data.csv")
y1_test = pd.read_csv("/Users/kellyt/Documents/ml/HW2/test1_icu_label.csv")

In [4]:
# print("x1_train:", x1_train.shape)
print("y1_train:", y1_train.shape)

print("x1_test:", x1_test.shape)
print("y1_test:", y1_test.shape)

# print("training distribution", y1_train.value_counts(normalize=True))
# print("testing distribution", y1_test.value_counts(normalize=True))

y1_train: (5000, 1)
x1_test: (1097, 108)
y1_test: (1097, 1)


In [5]:
# A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().

y1_train = y1_train.values.ravel()
y1_test = y1_test.values.ravel()

# Build SVM Model

## (1) Compare linear kernel and a nonlinear kernel (e.g., RBF) - discuss what the comparison tells you about the structure of the data

In [ ]:
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
results = []

for kernel in kernels:
    print(f"Training SVM with {kernel} kernel...")
    svm_model1 = SVC(kernel=kernel, random_state=42)
    X_train, X_test = x1_train, x1_test

    svm_model1.fit(X_train, y1_train)
    preds = svm_model1.predict(X_test)

    # for (3)
    total_number_of_support_vectors = len(svm_model1.support_)
    number_of_support_vectors_per_class = svm_model1.n_support_

    results.append({
            'Experiment': 'Kernel SVM',
            'Kernel': kernel,
            'C': 1.0,
            'Accuracy': round(accuracy_score(y1_test, preds), 4),
            'Precision': round(precision_score(y1_test, preds),4 ),
            'Recall': round(recall_score(y1_test, preds),4 ),
            'F1-Score': round(f1_score(y1_test, preds), 4),
            'AUC-ROC': round(roc_auc_score(y1_test, preds), 4),
            'Total Num. of SV': total_number_of_support_vectors,
            'Num. of SV per class': number_of_support_vectors_per_class,
        })

# took 8mins to run

Training SVM with linear kernel...
Training SVM with rbf kernel...
Training SVM with poly kernel...
Training SVM with sigmoid kernel...


In [7]:
results

[{'Experiment': 'Kernel SVM',
  'Kernel': 'linear',
  'C': 1.0,
  'Accuracy': 0.7803,
  'Precision': 0.7931,
  'Recall': 0.7569,
  'F1-Score': 0.7746,
  'AUC-ROC': np.float64(0.7802),
  'Total Num. of SV': 2156,
  'Num. of SV per class': array([1075, 1081], dtype=int32)},
 {'Experiment': 'Kernel SVM',
  'Kernel': 'rbf',
  'C': 1.0,
  'Accuracy': 0.7302,
  'Precision': 0.7466,
  'Recall': 0.6947,
  'F1-Score': 0.7197,
  'AUC-ROC': np.float64(0.7301),
  'Total Num. of SV': 3602,
  'Num. of SV per class': array([1802, 1800], dtype=int32)},
 {'Experiment': 'Kernel SVM',
  'Kernel': 'poly',
  'C': 1.0,
  'Accuracy': 0.7238,
  'Precision': 0.7163,
  'Recall': 0.7386,
  'F1-Score': 0.7273,
  'AUC-ROC': np.float64(0.7238),
  'Total Num. of SV': 3644,
  'Num. of SV per class': array([1818, 1826], dtype=int32)},
 {'Experiment': 'Kernel SVM',
  'Kernel': 'sigmoid',
  'C': 1.0,
  'Accuracy': 0.4804,
  'Precision': 0.4799,
  'Recall': 0.5027,
  'F1-Score': 0.4911,
  'AUC-ROC': np.float64(0.4805),
 

## (2) Observe the effect of the regularization parameter C and of feature scaling on results

In [8]:
c_values = [0.01, 0.1, 1, 10]

for kernel in kernels:
    for c in c_values:
        for scaling in [True, False]:
            if scaling:
                print(f"Training SVM with scaled {kernel} kernel with parameter C {c}...")
                svm_model2 = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=c, random_state=42))
            else:
                print(f"Training SVM with {kernel} kernel with parameter C {c}...")
                svm_model2 = SVC(kernel=kernel, C=c, random_state=42)

            svm_model2.fit(x1_train, y1_train)
            preds = svm_model2.predict(x1_test)

            # for (3)
            svc_step = svm_model2.named_steps['svc'] if scaling else svm_model2
            
            total_number_of_support_vectors = len(svc_step.support_)
            number_of_support_vectors_per_class = list(svc_step.n_support_)

            results.append({
                    'Experiment': 'C Tuning',
                    'Kernel': kernel,
                    'C': c,
                    'Scaled': scaling,
                    'Accuracy': round(accuracy_score(y1_test, preds), 4),
                    'Precision': round(precision_score(y1_test, preds),4 ),
                    'Recall': round(recall_score(y1_test, preds),4 ),
                    'F1-Score': round(f1_score(y1_test, preds), 4),
                    'AUC-ROC': round(roc_auc_score(y1_test, preds), 4),
                    'Total Num. of SV': total_number_of_support_vectors,
                    'Num. of SV per class': number_of_support_vectors_per_class,
                })

# 

Training SVM with scaled linear kernel with parameter C 0.01...
Training SVM with linear kernel with parameter C 0.01...
Training SVM with scaled linear kernel with parameter C 0.1...
Training SVM with linear kernel with parameter C 0.1...
Training SVM with scaled linear kernel with parameter C 1...
Training SVM with linear kernel with parameter C 1...
Training SVM with scaled linear kernel with parameter C 10...
Training SVM with linear kernel with parameter C 10...
Training SVM with scaled rbf kernel with parameter C 0.01...
Training SVM with rbf kernel with parameter C 0.01...
Training SVM with scaled rbf kernel with parameter C 0.1...
Training SVM with rbf kernel with parameter C 0.1...
Training SVM with scaled rbf kernel with parameter C 1...
Training SVM with rbf kernel with parameter C 1...
Training SVM with scaled rbf kernel with parameter C 10...
Training SVM with rbf kernel with parameter C 10...
Training SVM with scaled poly kernel with parameter C 0.01...
Training SVM with 

In [9]:
results

[{'Experiment': 'Kernel SVM',
  'Kernel': 'linear',
  'C': 1.0,
  'Accuracy': 0.7803,
  'Precision': 0.7931,
  'Recall': 0.7569,
  'F1-Score': 0.7746,
  'AUC-ROC': np.float64(0.7802),
  'Total Num. of SV': 2156,
  'Num. of SV per class': array([1075, 1081], dtype=int32)},
 {'Experiment': 'Kernel SVM',
  'Kernel': 'rbf',
  'C': 1.0,
  'Accuracy': 0.7302,
  'Precision': 0.7466,
  'Recall': 0.6947,
  'F1-Score': 0.7197,
  'AUC-ROC': np.float64(0.7301),
  'Total Num. of SV': 3602,
  'Num. of SV per class': array([1802, 1800], dtype=int32)},
 {'Experiment': 'Kernel SVM',
  'Kernel': 'poly',
  'C': 1.0,
  'Accuracy': 0.7238,
  'Precision': 0.7163,
  'Recall': 0.7386,
  'F1-Score': 0.7273,
  'AUC-ROC': np.float64(0.7238),
  'Total Num. of SV': 3644,
  'Num. of SV per class': array([1818, 1826], dtype=int32)},
 {'Experiment': 'Kernel SVM',
  'Kernel': 'sigmoid',
  'C': 1.0,
  'Accuracy': 0.4804,
  'Precision': 0.4799,
  'Recall': 0.5027,
  'F1-Score': 0.4911,
  'AUC-ROC': np.float64(0.4805),
 

## (3) Look at the number of support vectors under different settings and discuss what it reflects.

In [10]:
results_df = pd.DataFrame(results)
results_df

,Experiment,Kernel,C,Accuracy,Precision,Recall,F1-Score,AUC-ROC,Total Num. of SV,Num. of SV per class,Scaled
0,Kernel SVM,linear,1.00,0.7803,0.7931,0.7569,0.7746,0.7802,2156,"[1075, 1081]",NaN
1,Kernel SVM,rbf,1.00,0.7302,0.7466,0.6947,0.7197,0.7301,3602,"[1802, 1800]",NaN
2,Kernel SVM,poly,1.00,0.7238,0.7163,0.7386,0.7273,0.7238,3644,"[1818, 1826]",NaN
3,Kernel SVM,sigmoid,1.00,0.4804,0.4799,0.5027,0.4911,0.4805,2833,"[1416, 1417]",NaN
4,C Tuning,linear,0.01,0.7876,0.7951,0.7733,0.7841,0.7876,2511,"[1257, 1254]",True
5,C Tuning,linear,0.01,0.7858,0.7868,0.7824,0.7846,0.7858,2561,"[1278, 1283]",False
6,C Tuning,linear,0.10,0.7940,0.8034,0.7770,0.7900,0.7939,2417,"[1211, 1206]",True
7,C Tuning,linear,0.10,0.7885,0.7966,0.7733,0.7848,0.7885,2438,"[1223, 1215]",False
8,C Tuning,linear,1.00,0.7931,0.8030,0.7751,0.7888,0.7930,2405,"[1205, 1200]",True
9,C Tuning,linear,1.00,0.7803,0.7931,0.7569,0.7746,0.7802,2156,"[1075, 1081]",False
